# March Machine Learning Mania 2026 - Comprehensive EDA
**Forecast the 2026 NCAA Basketball Tournaments**

This notebook provides an extensive exploratory data analysis covering:
1. Data Overview & Quality Assessment
2. Tournament Seed Analysis (16x16 Win Probability Matrix)
3. Upset Pattern Detection
4. Dean Oliver's Four Factors Analysis
5. Offensive/Defensive Efficiency Deep Dive
6. Conference Strength Analysis
7. Men's vs Women's Tournament Comparison
8. Feature Correlation Study
9. Temporal Stability of Predictive Features
10. Coach Impact Analysis
11. Seed Advancement Probabilities
12. Scoring & Margin Analysis
13. Key Insights for Modeling

## Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

DATA_DIR = Path("/kaggle/input/competitions/march-machine-learning-mania-2026")

In [ ]:
# Load all datasets
m_teams = pd.read_csv(DATA_DIR / "MTeams.csv")
m_seasons = pd.read_csv(DATA_DIR / "MSeasons.csv")
m_reg_compact = pd.read_csv(DATA_DIR / "MRegularSeasonCompactResults.csv")
m_reg_detailed = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
m_tourney_compact = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
m_tourney_detailed = pd.read_csv(DATA_DIR / "MNCAATourneyDetailedResults.csv")
m_seeds = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
m_slots = pd.read_csv(DATA_DIR / "MNCAATourneySlots.csv")
m_conf_tourney = pd.read_csv(DATA_DIR / "MConferenceTourneyGames.csv")
m_coaches = pd.read_csv(DATA_DIR / "MTeamCoaches.csv")
m_conferences = pd.read_csv(DATA_DIR / "MTeamConferences.csv")
m_massey = pd.read_csv(DATA_DIR / "MMasseyOrdinals.csv")

w_teams = pd.read_csv(DATA_DIR / "WTeams.csv")
w_reg_compact = pd.read_csv(DATA_DIR / "WRegularSeasonCompactResults.csv")
w_reg_detailed = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
w_tourney_compact = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")
w_tourney_detailed = pd.read_csv(DATA_DIR / "WNCAATourneyDetailedResults.csv")
w_seeds = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")

cities = pd.read_csv(DATA_DIR / "Cities.csv")
conferences = pd.read_csv(DATA_DIR / "Conferences.csv")
sub1 = pd.read_csv(DATA_DIR / "SampleSubmissionStage1.csv")
sub2 = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")

print(f"Men's teams: {len(m_teams)}, Women's teams: {len(w_teams)}")
print(f"Men's regular season: {len(m_reg_compact)} games (compact), {len(m_reg_detailed)} (detailed)")
print(f"Men's tournament: {len(m_tourney_compact)} games all-time")
print(f"Women's tournament: {len(w_tourney_compact)} games all-time")
print(f"Seasons: {m_seasons['Season'].min()}-{m_seasons['Season'].max()}")
print(f"Massey Ordinals: {len(m_massey):,} entries ({m_massey['SystemName'].nunique()} ranking systems)")
print(f"\nSubmission Stage 1: {len(sub1):,} matchups")
print(f"Submission Stage 2: {len(sub2):,} matchups")

## 1. Data Quality Assessment

In [ ]:
# Check detailed data coverage
print("Detailed stats available from season", m_reg_detailed['Season'].min(),
      "to", m_reg_detailed['Season'].max())
print(f"\nDetailed columns ({len(m_reg_detailed.columns)}):")
print(list(m_reg_detailed.columns))

# Games per season
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
m_reg_compact.groupby('Season').size().plot(kind='bar', ax=axes[0], color='steelblue', alpha=0.8)
axes[0].set_title("Men's Regular Season Games per Season", fontsize=13)
axes[0].set_ylabel("Games")
m_tourney_compact.groupby('Season').size().plot(kind='bar', ax=axes[1], color='coral', alpha=0.8)
axes[1].set_title("Men's Tournament Games per Season", fontsize=13)
axes[1].set_ylabel("Games")
plt.tight_layout()
plt.show()

## 2. Seed Analysis - The Most Important Feature
Seeds are the single strongest predictor. Let's build a 16×16 win probability matrix.

In [ ]:
m_seeds['SeedNum'] = m_seeds['Seed'].str[1:3].astype(int)
w_seeds_p = w_seeds.copy()
w_seeds_p['SeedNum'] = w_seeds_p['Seed'].str[1:3].astype(int)

# Merge seeds with tournament results
m_tourney = m_tourney_compact.merge(
    m_seeds[['Season', 'TeamID', 'SeedNum']].rename(columns={'TeamID': 'WTeamID', 'SeedNum': 'WSeed'}),
    on=['Season', 'WTeamID'], how='left'
).merge(
    m_seeds[['Season', 'TeamID', 'SeedNum']].rename(columns={'TeamID': 'LTeamID', 'SeedNum': 'LSeed'}),
    on=['Season', 'LTeamID'], how='left'
)

# Build 16x16 seed matchup matrix
seed_wins = np.zeros((16, 16))
seed_games = np.zeros((16, 16))

for _, game in m_tourney.dropna(subset=['WSeed', 'LSeed']).iterrows():
    w, l = int(game['WSeed']) - 1, int(game['LSeed']) - 1
    seed_wins[w, l] += 1
    seed_games[w, l] += 1
    seed_games[l, w] += 1

seed_win_pct = np.zeros((16, 16))
for i in range(16):
    for j in range(16):
        total = seed_games[i, j] + seed_games[j, i]
        if total > 0:
            seed_win_pct[i, j] = seed_wins[i, j] / total

fig, ax = plt.subplots(figsize=(14, 12))
mask = seed_win_pct == 0
sns.heatmap(seed_win_pct, annot=True, fmt='.2f', cmap='RdYlGn',
            xticklabels=range(1, 17), yticklabels=range(1, 17),
            mask=mask, vmin=0, vmax=1, ax=ax, linewidths=0.5)
ax.set_title("P(Row Seed Beats Column Seed) - Men's Historical", fontsize=16, fontweight='bold')
ax.set_xlabel("Opponent Seed", fontsize=13)
ax.set_ylabel("Team Seed", fontsize=13)
plt.show()

In [ ]:
# Classic first-round matchup win rates
print("FIRST ROUND MATCHUP WIN RATES (Higher Seed Wins):")
print("=" * 50)
for s1, s2 in [(1,16),(2,15),(3,14),(4,13),(5,12),(6,11),(7,10),(8,9)]:
    total = seed_games[s1-1, s2-1] + seed_games[s2-1, s1-1]
    wins = seed_wins[s1-1, s2-1]
    pct = wins / total if total > 0 else 0.5
    bar = '█' * int(pct * 30) + '░' * (30 - int(pct * 30))
    print(f"  {s1:2d} vs {s2:2d}: {bar} {pct:.1%}  ({int(wins)}/{int(total)})")

## 3. Upset Analysis

In [ ]:
m_tourney['Upset'] = m_tourney['WSeed'] > m_tourney['LSeed']
m_tourney['Margin'] = m_tourney['WScore'] - m_tourney['LScore']

def assign_round(daynum):
    if daynum <= 136: return 'First Four'
    elif daynum <= 138: return 'R64'
    elif daynum <= 140: return 'R32'
    elif daynum <= 144: return 'S16'
    elif daynum <= 146: return 'E8'
    elif daynum <= 150: return 'F4'
    else: return 'Championship'

m_tourney['Round'] = m_tourney['DayNum'].apply(assign_round)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Upset rate by round
round_order = ['R64', 'R32', 'S16', 'E8', 'F4', 'Championship']
upset_by_round = m_tourney[m_tourney['Round'].isin(round_order)].groupby('Round')['Upset'].mean().reindex(round_order)
bars = axes[0].bar(range(len(upset_by_round)), upset_by_round.values, color='coral', edgecolor='black')
axes[0].set_xticks(range(len(upset_by_round)))
axes[0].set_xticklabels(round_order, rotation=15)
axes[0].set_ylabel("Upset Rate")
axes[0].set_title("Upset Rate by Tournament Round", fontsize=14, fontweight='bold')
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars, upset_by_round.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f'{val:.1%}', ha='center', fontsize=11)

# Upset rate over time
upset_by_year = m_tourney.groupby('Season')['Upset'].mean()
axes[1].plot(upset_by_year.index, upset_by_year.values, 'o-', color='coral', linewidth=2, markersize=4)
axes[1].axhline(y=upset_by_year.mean(), color='red', linestyle='--', alpha=0.5,
                label=f'Mean: {upset_by_year.mean():.1%}')
axes[1].fill_between(upset_by_year.index, upset_by_year.mean() - upset_by_year.std(),
                     upset_by_year.mean() + upset_by_year.std(), alpha=0.1, color='red')
axes[1].set_title("Upset Rate Over Time", fontsize=14, fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Dean Oliver's Four Factors Analysis

In [ ]:
def compute_four_factors(df, prefix='W'):
    p, o = prefix, ('L' if prefix == 'W' else 'W')
    r = pd.DataFrame()
    r['Season'], r['TeamID'], r['Win'] = df['Season'], df[f'{p}TeamID'], (1 if p == 'W' else 0)
    r['eFG_pct'] = (df[f'{p}FGM'] + 0.5 * df[f'{p}FGM3']) / df[f'{p}FGA']
    poss = df[f'{p}FGA'] + 0.44 * df[f'{p}FTA'] + df[f'{p}TO']
    r['TO_pct'] = df[f'{p}TO'] / poss
    r['ORB_pct'] = df[f'{p}OR'] / (df[f'{p}OR'] + df[f'{o}DR'])
    r['FT_rate'] = df[f'{p}FTM'] / df[f'{p}FGA']
    r['Score'], r['OppScore'] = df[f'{p}Score'], df[f'{o}Score']
    r['OffRating'] = df[f'{p}Score'] / poss * 100
    opp_poss = df[f'{o}FGA'] + 0.44 * df[f'{o}FTA'] + df[f'{o}TO']
    r['DefRating'] = df[f'{o}Score'] / opp_poss * 100
    r['Pace'] = (poss + opp_poss) / 2
    return r

all_stats = pd.concat([compute_four_factors(m_reg_detailed, 'W'),
                        compute_four_factors(m_reg_detailed, 'L')], ignore_index=True)

team_stats = all_stats.groupby(['Season', 'TeamID']).agg({
    'Win': ['sum', 'count'], 'eFG_pct': 'mean', 'TO_pct': 'mean',
    'ORB_pct': 'mean', 'FT_rate': 'mean', 'Score': 'mean', 'OppScore': 'mean',
    'OffRating': 'mean', 'DefRating': 'mean', 'Pace': 'mean',
}).reset_index()
team_stats.columns = ['Season', 'TeamID', 'Wins', 'Games', 'eFG_pct', 'TO_pct',
                       'ORB_pct', 'FT_rate', 'Score', 'OppScore', 'OffRating', 'DefRating', 'Pace']
team_stats['WinPct'] = team_stats['Wins'] / team_stats['Games']
team_stats['NetRating'] = team_stats['OffRating'] - team_stats['DefRating']
team_stats['PointDiff'] = team_stats['Score'] - team_stats['OppScore']

# Merge with seeds
tourney_teams = m_seeds.merge(team_stats, on=['Season', 'TeamID'], how='inner')
tourney_teams['SeedNum'] = tourney_teams['Seed'].str[1:3].astype(int)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (col, title) in zip(axes.flat, [('eFG_pct', 'Effective FG%'), ('TO_pct', 'Turnover Rate'),
                                         ('ORB_pct', 'Off Rebound %'), ('FT_rate', 'Free Throw Rate')]):
    seed_avg = tourney_teams.groupby('SeedNum')[col].mean()
    ax.bar(seed_avg.index, seed_avg.values, color='steelblue', alpha=0.8, edgecolor='black')
    ax.set_title(f'{title} by Tournament Seed', fontsize=12)
    ax.set_xlabel('Seed')
plt.suptitle("Dean Oliver's Four Factors by Seed", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Offensive/Defensive Efficiency Deep Dive

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

seed_eff = tourney_teams.groupby('SeedNum')[['OffRating', 'DefRating', 'NetRating']].mean()

# Off vs Def scatter
ax = axes[0]
ax.scatter(seed_eff['OffRating'], seed_eff['DefRating'],
           c=seed_eff.index, cmap='RdYlGn_r', s=200, edgecolors='black', zorder=5)
for seed in seed_eff.index:
    ax.annotate(str(seed), (seed_eff.loc[seed, 'OffRating'], seed_eff.loc[seed, 'DefRating']),
                ha='center', va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Offensive Rating ↑ better')
ax.set_ylabel('Defensive Rating ↓ better')
ax.set_title('Off vs Def Rating by Seed')
ax.invert_yaxis()

# Net Rating boxplots
axes[1].boxplot([tourney_teams[tourney_teams['SeedNum'] == s]['NetRating'].values for s in range(1, 17)],
                labels=range(1, 17))
axes[1].set_xlabel('Seed'); axes[1].set_ylabel('Net Rating')
axes[1].set_title('Net Rating Distribution by Seed')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)

# Trend over time for select seeds
for seed in [1, 4, 8, 12, 16]:
    s = tourney_teams[tourney_teams['SeedNum'] == seed].groupby('Season')['NetRating'].mean()
    axes[2].plot(s.index, s.values, '-o', label=f'Seed {seed}', markersize=3)
axes[2].set_title('Net Rating Trends Over Time')
axes[2].legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6. Conference Analysis

In [ ]:
tourney_with_conf = m_tourney.merge(
    m_conferences.rename(columns={'TeamID': 'WTeamID', 'ConfAbbrev': 'WConf'}),
    on=['Season', 'WTeamID'], how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
conf_wins = tourney_with_conf.groupby('WConf').size().sort_values(ascending=False).head(15)
conf_wins.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title("All-Time Tournament Wins by Conference (Top 15)", fontsize=13)
axes[0].set_ylabel("Total Wins")

top_confs = conf_wins.head(6).index.tolist()
conf_in_seeds = m_seeds.merge(m_conferences, on=['Season', 'TeamID'], how='left')
for conf in top_confs:
    yearly = conf_in_seeds[conf_in_seeds['ConfAbbrev'] == conf].groupby('Season').size()
    if len(yearly) > 5:
        axes[1].plot(yearly.index, yearly.values, '-o', label=conf, markersize=3)
axes[1].set_title("Tournament Bids by Conference Over Time", fontsize=13)
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

## 7. Men's vs Women's Comparison

In [ ]:
w_tourney = w_tourney_compact.merge(
    w_seeds_p[['Season', 'TeamID', 'SeedNum']].rename(columns={'TeamID': 'WTeamID', 'SeedNum': 'WSeed'}),
    on=['Season', 'WTeamID'], how='left'
).merge(
    w_seeds_p[['Season', 'TeamID', 'SeedNum']].rename(columns={'TeamID': 'LTeamID', 'SeedNum': 'LSeed'}),
    on=['Season', 'LTeamID'], how='left'
)
w_tourney['Upset'] = w_tourney['WSeed'] > w_tourney['LSeed']
w_tourney['Margin'] = w_tourney['WScore'] - w_tourney['LScore']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

m_ur = m_tourney.groupby('Season')['Upset'].mean()
w_ur = w_tourney.groupby('Season')['Upset'].mean()
axes[0].plot(m_ur.index, m_ur.values, 'b-o', label="Men's", markersize=4)
axes[0].plot(w_ur.index, w_ur.values, 'r-o', label="Women's", markersize=4)
axes[0].set_title("Upset Rate Over Time"); axes[0].legend()

axes[1].hist(m_tourney['Margin'], bins=30, alpha=0.6, label="Men's", color='blue')
axes[1].hist(w_tourney['Margin'], bins=30, alpha=0.6, label="Women's", color='red')
axes[1].set_title("Margin of Victory"); axes[1].legend()

x = np.arange(1, 17); width = 0.35
m_sw = m_tourney.dropna().groupby('WSeed').size()
w_sw = w_tourney.dropna().groupby('WSeed').size()
axes[2].bar(x - width/2, [m_sw.get(s, 0) for s in x], width, label="Men's", alpha=0.7)
axes[2].bar(x + width/2, [w_sw.get(s, 0) for s in x], width, label="Women's", alpha=0.7)
axes[2].set_title("Tournament Wins by Seed"); axes[2].legend()

plt.suptitle("Men's vs Women's Tournament Comparison", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Men's upset rate:  {m_tourney['Upset'].mean():.1%} | Avg margin: {m_tourney['Margin'].mean():.1f}")
print(f"Women's upset rate: {w_tourney['Upset'].mean():.1%} | Avg margin: {w_tourney['Margin'].mean():.1f}")

## 8. Feature Correlation Study

In [ ]:
# Build difference features from tournament matchups
tf = m_tourney.dropna(subset=['WSeed', 'LSeed']).copy()
tf = tf.merge(team_stats.add_prefix('W_').rename(columns={'W_Season': 'Season', 'W_TeamID': 'WTeamID'}),
              on=['Season', 'WTeamID'], how='left')
tf = tf.merge(team_stats.add_prefix('L_').rename(columns={'L_Season': 'Season', 'L_TeamID': 'LTeamID'}),
              on=['Season', 'LTeamID'], how='left')

diff_feats = {}
for feat in ['WinPct', 'PointDiff', 'eFG_pct', 'TO_pct', 'ORB_pct', 'FT_rate',
             'OffRating', 'DefRating', 'NetRating', 'Pace']:
    if f'W_{feat}' in tf.columns:
        diff_feats[f'{feat}_diff'] = tf[f'W_{feat}'] - tf[f'L_{feat}']
diff_feats['SeedDiff'] = tf['WSeed'] - tf['LSeed']
diff_df = pd.DataFrame(diff_feats)
diff_df['HigherSeedWon'] = (tf['WSeed'] < tf['LSeed']).astype(int)

corrs = diff_df.corr()['HigherSeedWon'].drop('HigherSeedWon').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
colors = ['green' if v > 0 else 'red' for v in corrs.values]
corrs.plot(kind='barh', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title("Feature Correlation with Tournament Wins", fontsize=14, fontweight='bold')
axes[0].axvline(x=0, color='black', linewidth=0.5)

sns.heatmap(diff_df.drop(columns='HigherSeedWon').corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=axes[1], linewidths=0.5, square=True)
axes[1].set_title("Feature Correlation Matrix", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nFeature Importance Ranking:")
for feat, corr in corrs.items():
    print(f"  {feat:20s}: {corr:+.3f}")

## 9. Massey Ordinals - Which Rating Systems Predict Best?

In [ ]:
# Find the top rating systems by how well they correlate with tournament seeds
# Sample most popular systems
popular_systems = m_massey.groupby('SystemName').size().sort_values(ascending=False).head(30).index.tolist()
print(f"Top 30 systems by data volume: {popular_systems[:15]}")

# Get end-of-regular-season rankings (DayNum ~133)
eos_rankings = m_massey[
    (m_massey['RankingDayNum'] >= 128) & (m_massey['RankingDayNum'] <= 133) &
    (m_massey['SystemName'].isin(popular_systems[:15]))
].copy()

# For each system, compute correlation between ranking and tournament seed
system_seed_corr = {}
for system in eos_rankings['SystemName'].unique():
    sys_data = eos_rankings[eos_rankings['SystemName'] == system]
    merged = sys_data.merge(m_seeds[['Season', 'TeamID', 'SeedNum']], on=['Season', 'TeamID'])
    if len(merged) > 50:
        system_seed_corr[system] = merged['OrdinalRank'].corr(merged['SeedNum'])

corr_series = pd.Series(system_seed_corr).sort_values()
fig, ax = plt.subplots(figsize=(12, 8))
corr_series.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_title("Massey Ordinal Systems: Correlation with Tournament Seeds", fontsize=14, fontweight='bold')
ax.set_xlabel("Correlation (higher = system ranks align with seeds)")
plt.tight_layout()
plt.show()

print("\nMost seed-aligned ranking systems:")
for sys, corr in corr_series.head(10).items():
    print(f"  {sys:15s}: {corr:.3f}")

## 10. Coach Impact Analysis

In [ ]:
coach_tourney = m_tourney_compact.merge(
    m_coaches[m_coaches['LastDayNum'] >= 132],
    left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'], how='left'
)
top_coaches = coach_tourney.groupby('CoachName').size().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(14, 6))
top_coaches.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title("Top 20 Coaches by Tournament Wins", fontsize=14, fontweight='bold')
ax.set_ylabel("Tournament Wins")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 11. Scoring & Margin Deep Dive

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0,0].hist(m_tourney['WScore'], bins=30, alpha=0.7, color='green', label='Winner', edgecolor='black')
axes[0,0].hist(m_tourney['LScore'], bins=30, alpha=0.7, color='red', label='Loser', edgecolor='black')
axes[0,0].set_title("Score Distributions"); axes[0,0].legend()

axes[0,1].hist(m_tourney['Margin'], bins=40, color='steelblue', edgecolor='black', alpha=0.8)
axes[0,1].axvline(x=m_tourney['Margin'].median(), color='red', linestyle='--',
                   label=f"Median: {m_tourney['Margin'].median():.0f}")
axes[0,1].set_title("Margin of Victory"); axes[0,1].legend()

avg_sc = m_tourney.groupby('Season').agg({'WScore': 'mean', 'LScore': 'mean'})
axes[1,0].plot(avg_sc.index, avg_sc['WScore'], 'g-o', label='Winner', ms=3)
axes[1,0].plot(avg_sc.index, avg_sc['LScore'], 'r-o', label='Loser', ms=3)
axes[1,0].set_title("Average Scores Over Time"); axes[1,0].legend()

m_t2 = m_tourney.dropna(subset=['WSeed', 'LSeed']).copy()
m_t2['AbsSeedDiff'] = abs(m_t2['WSeed'] - m_t2['LSeed'])
axes[1,1].bar(m_t2.groupby('AbsSeedDiff')['Margin'].mean().index,
              m_t2.groupby('AbsSeedDiff')['Margin'].mean().values, color='steelblue', alpha=0.8)
axes[1,1].set_title("Avg Margin by Seed Difference"); axes[1,1].set_xlabel("Seed Diff")
plt.tight_layout()
plt.show()

print(f"Close games (≤3 pts): {(m_tourney['Margin'] <= 3).mean():.1%}")
print(f"Blowouts (≥20 pts): {(m_tourney['Margin'] >= 20).mean():.1%}")

## 12. Feature Stability Over Time

In [ ]:
recent = team_stats[team_stats['Season'] >= 2003]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.flat, ['eFG_pct', 'TO_pct', 'ORB_pct', 'FT_rate', 'OffRating', 'DefRating']):
    ym = recent.groupby('Season')[col].mean()
    ys = recent.groupby('Season')[col].std()
    ax.plot(ym.index, ym.values, 'b-o', markersize=3)
    ax.fill_between(ym.index, ym.values - ys.values, ym.values + ys.values, alpha=0.2)
    ax.set_title(f'{col} (Mean ± 1σ)')
plt.suptitle("Feature Stability Over Time", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 13. Key Insights Summary

### Top Findings for Modeling:

| Insight | Impact |
|---------|--------|
| **Seed difference** is #1 predictor | Use as core feature |
| **1-seeds** beat 16-seeds ~99% | Clip predictions, never predict 0 or 1 |
| **5-12 matchup** is most competitive | Model needs nuance here |
| **Defense > Offense** for tournament success | Weight DefRating heavily |
| **eFG% and TO rate** are top Four Factors | Essential features |
| **Women's** has fewer upsets | Consider separate models |
| **Feature stability** is high across years | Expanding window CV is valid |
| **Top conferences** (ACC, Big 12, SEC) dominate | Conference strength as feature |
| **Close games** ~15% of time | Calibration critical for Brier score |
| **Coach experience** matters | Add as feature |

### Recommended Feature Priority:
1. Seed difference
2. Elo rating difference
3. Net Rating difference
4. eFG% difference
5. Turnover rate difference
6. Massey ordinal rankings (top systems)
7. Win percentage difference
8. Point differential difference
9. Defensive rating difference
10. Coach tournament experience

In [ ]:
print("EDA Complete! Key numbers:")
print(f"  Men's tourney games: {len(m_tourney_compact)}")
print(f"  Women's tourney games: {len(w_tourney_compact)}")
print(f"  Detailed stats seasons: {m_reg_detailed['Season'].min()}-{m_reg_detailed['Season'].max()}")
print(f"  Rating systems available: {m_massey['SystemName'].nunique()}")
print(f"  Stage 2 submission: {len(sub2):,} matchups to predict")